In [ ]:
# Ugly Automatic processing of image stacks
recons_name = []  # for gifs
fth_reco_name = []
images = []
energy_list = []

for it, im_id in enumerate(tqdm(im_id_set,desc="im_ids")):
    for stack in tqdm(range(get_marana_filenames(im_id, file_indices = None).shape[0]),desc="marana files"):
        # Load energy and add to experimental setup
        experimental_setup["energy"] = load_key(im_id, "/scan/instrument/collection/mono")
        experimental_setup["lambda"] = helper.photon_energy_wavelength(
            experimental_setup["energy"], input_unit="eV"
        )
        
        # Load images
        #image, _ = load_processing(im_id, file_indices=np.arange(4)+4*stack)
        image, _ = load_processing(im_id, file_indices=[stack])
    
        # Load dark
        dark_id = dark_id_im_set[it]
        if dark_id is not None:
            dark = dark_all.copy()
            #, _ = load_processing(dark_id, crop=None)
            image = image - dark
    
        # Get topo
        topo_id = topo_id_set[it]
    
        # Load data
        if isinstance(topo_id_set[0], list):
            topo_id = topo_id[0]
        else:
            print(f"Loading imageId: %04d, topoId: %04d" % (im_id, topo_id))
        topo = topo_all.copy()
    
        # Process images
        worker_dict = worker(image, topo, Norm=True)
    
        # Save topo hologram
        #save_topo_holo(worker_dict["sum_c"], im_id, topo_id)   
        
        # Reconstruct
        recon = worker_dict["recon"]
        
        # Plot
        #title = get_title("magnet_A", im_id, topo_id,  Framenumber= stack)
        #plot_recon(recon[roi_cdi_s], title)
    
        if use_sub_frame:
            # Save images
            fname = join(
            folder_general,
            "Recon_ImId_%04d_RefId_%s_fth_diff_stack_%s_frame_%s.png" % (im_id, topo_id, USER, it),
            )
        else: 
            # Save images
            fname = join(
                folder_general,
                "Recon_ImId_%04d_RefId_%s_fth_diff_stack_%s.png" % (im_id, topo_id, USER),
            )
        #print("Saving: %s" % fname)
        #plt.savefig(fname, bbox_inches="tight", transparent=False)
        #fth_reco_name.append(fname)
        # Optional: Save hdf5 file of fth data
        #save_fth_h5()
    
        ################ CDI ###############
        # Create pos and neg helicity set
        pos = (worker_dict["sum_c"] + worker_dict["diff_c"])/2
        neg = (worker_dict["sum_c"] - worker_dict["diff_c"])/2
    
        # Create mask pixel
        mask_pixel = worker_dict["mask_pixel"]
    
        # Create supportmask from coordinates
        supportmask = mask_lib.create_circle_supportmask(support_coordinates, pos.shape)
        
        if stack == 0:
            # Do phase retrieval
            (
                retrieved_p,
                retrieved_n,
                retrieved_p_pc,
                retrieved_n_pc,
                bsmask_p,
                bsmask_n,
                gamma_p,
                gamma_n,
            ) = phase_retrieval(pos, neg, mask_pixel, supportmask, vmin = offset_vmin, Startimage=Startimage, Startgamma=Startgamma)
            retrieved_p_pc_start, gamma_p_start, retrieved_n_pc_start, gamma_n_start = retrieved_p_pc, gamma_p, retrieved_n_pc, gamma_n
        elif stack>0:
            # Do phase retrieval
            offset_vmin = 0.5
            mi, _ = np.percentile(pos[pos != 0], [offset_vmin, 99.9])
            pos3 = pos - mi
            (
                retrieved_p_pc,
                Error_diff_p_pc2,
                Error_supp,
                gamma_p,
            ) = PhR.PhaseRtrv_with_RL(
                diffract=np.sqrt(np.maximum(pos3, np.zeros(pos3.shape))),
                mask=supportmask,
                mode="mine",
                beta_zero=0.5,
                Nit=10,
                beta_mode="const",
                gamma=gamma_n_start.copy(),
                RL_freq=20,
                RL_it=50,
                Phase=retrieved_n_pc_start.copy(),
                real_object=False,
                bsmask=mask_pixel.astype(int),
                average_img=30,
                Fourier_last=True,
            )
        # Get Recos partial coherence
        # Positiv partial coherence
        p_pc = fth.reconstructCDI(
            fth.propagate(
                retrieved_p_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )
        # Negative partial coherence
        n_pc = fth.reconstructCDI(
            fth.propagate(
                retrieved_n_pc * mask_bs_cdi,
                prop_dist_cdi * 1e-6,
                experimental_setup=experimental_setup,
            )
        )
    
        ##### Calc reco and optimze contrast
        recon = helicity[it] * (p_pc - n_pc)
        #_, phase_cdi = optimize_phase_contrast(recon, supportmask, method="contrast")
        #phase_cdi = 0
        recon = recon * np.exp(1j * phase_cdi)
        print("Phase is:", np.round(phase_cdi, 2))
        ########
    
        # Plot
        #try:
        #    title = get_title("magnet_A", im_id, topo_id, CDI=True,Framenumber=stack)
        #except:
        title = get_title(None, im_id, topo_id, CDI=True,Framenumber=stack)
        plot_recon(recon[roi_cdi_s], title)
        images.append(recon[roi_cdi_s])
        
        if use_sub_frame:
            fname = join(
            folder_general,
            "Recon_ImId_%04d_RefId_%s_cdi_stack_%s_frame_%s.png" % (im_id, topo_id, USER, stack),
            )
        else:
            fname = join(
                folder_general,
                "Recon_ImId_%04d_RefId_%s_cdi_stack_%s.png" % (im_id, topo_id, USER),
            )
    
        print("Saving: %s" % fname)
        plt.savefig(fname, bbox_inches="tight", transparent=False, dpi=300)
        recons_name.append(fname)
    
        # Save files as h5
        save_cdi_h5()
        print(" ")
print("CDI stack processing finished")